# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleeshanawal-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



My lane is Refresh and Content Opportunity Ranking. The main ML task type is ranking. The system will give each content page a priority score and then arrange the pages from highest to lowest refresh priority.

The output will help a content editor decide which pages should be reviewed first. The model is not automatically changing the content. It is supporting the editor's decision by producing a ranked review list.


In [6]:
import os
import sys
import subprocess
import pandas as pd

# Location of my GitHub repository
REPO_URL = "https://github.com/aleeshanawal-cmyk/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

# In Google Colab, download the repository if it is not already downloaded
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

# Load the starter dataset
data_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(
        "The starter dataset was not found. Check that the repository was cloned correctly."
    )

df = pd.read_csv(data_path)

# My lane focuses on pages with enough activity to support a useful decision
lane_df = df[
    (df["impressions_90d"] >= 100) &
    (df["sessions_90d"] > 0)
].copy()

print("Full dataset shape:", df.shape)
print("Lane dataframe shape:", lane_df.shape)
print("Task type: Ranking")
print("One row represents one content page.")

Full dataset shape: (30000, 44)
Lane dataframe shape: (22006, 44)
Task type: Ranking
One row represents one content page.


## 2. Target or proxy

The real target would be called `future_decline`. It would be 1 when a page's search impressions decline by more than 20% in a later 30-day period, and 0 when the page does not decline.

This future target should come from an observed later outcome. However, the starter dataset does not contain a completely separate future outcome window. Therefore, in this notebook I will create `is_declining_proxy` from the existing `trend_direction` column.

The proxy will be 1 when `trend_direction` is `down` and 0 otherwise. This is a defined proxy rather than a truly observed future label. Because the proxy comes from `trend_direction` and `trend_pct`, those two columns must not be used as model input features.

In [7]:
# Create a temporary proxy target for this framing exercise
lane_df["is_declining_proxy"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print("Proxy target counts:")
print(lane_df["is_declining_proxy"].value_counts())

print(
    "\nPercentage of lane pages marked as declining:",
    round(lane_df["is_declining_proxy"].mean() * 100, 2),
    "%"
)

# Show what the target column looks like
lane_df[
    [
        "content_id",
        "trend_direction",
        "is_declining_proxy"
    ]
].head(10)


Proxy target counts:
is_declining_proxy
1    13152
0     8854
Name: count, dtype: int64

Percentage of lane pages marked as declining: 59.77 %


,content_id,trend_direction,is_declining_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1
10,content_d8ee6cc6d642,stable,0




My main success metric will be Precision@50.

Precision@50 measures how many of the 50 pages ranked highest for refresh are actually declining in the later outcome period. This metric matches the real action because a content team may only have enough time to review a limited number of pages.

I will consider the first model useful if its Precision@50 is at least 0.70. This means that at least 35 of the top 50 recommended pages should be correct.

A high Precision@50 reduces wasted editor time. A wrong high-priority recommendation may cause an editor to spend time reviewing a page that did not need urgent attention.

In [8]:
TOP_K = 50
GOOD_PRECISION_AT_50 = 0.70

minimum_correct_pages = int(TOP_K * GOOD_PRECISION_AT_50)
proxy_base_rate = lane_df["is_declining_proxy"].mean()

print("Selected metric: Precision@50")
print("Target Precision@50:", GOOD_PRECISION_AT_50)
print("Minimum correct pages required:", minimum_correct_pages, "out of", TOP_K)
print("Current proxy declining rate:", round(proxy_base_rate, 3))


Selected metric: Precision@50
Target Precision@50: 0.7
Minimum correct pages required: 35 out of 50
Current proxy declining rate: 0.598




The unit of analysis is one anonymized content page. Therefore, one row in the dataframe represents one page.

For this lane, I use pages with at least 100 search impressions and more than zero sessions during the 90-day period. This gives the content team pages with enough measurable activity to support a refresh decision.

The dataframe includes page characteristics and performance signals such as content type, content age, time since the last update, impressions, clicks, sessions, click-through rate and average search position. It also shows the temporary decline proxy.

In [9]:
display_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "is_declining_proxy"
]

unit_of_analysis_df = lane_df[display_columns].copy()

print("Number of rows:", len(unit_of_analysis_df))
print("Number of columns shown:", len(display_columns))
print("Unit of analysis: one row = one content page")

unit_of_analysis_df.head(10)


Number of rows: 22006
Number of columns shown: 10
Unit of analysis: one row = one content page


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,is_declining_proxy
0,content_304f48230142,keyword article,187,20,3803,29,17,0.76,10.6,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,7,9,0.05,20.3,1
2,content_9aa793d4d895,keyword article,141,20,12581,11,11,0.09,36.5,1
3,content_331d6c4de07b,keyword article,463,22,11751,58,78,0.49,6.2,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,24,145,0.13,44.0,1
5,content_d4084a4bc775,keyword article,147,20,3970,1,5,0.03,8.5,1
7,content_a63219c6e95a,keyword article,445,22,1724,1,28,0.06,21.2,0
8,content_5e6c160719bc,keyword article,90,20,32574,29,68,0.09,46.0,1
9,content_c27558df2b0c,keyword article,257,104,1240,2,3,0.16,4.9,1
10,content_d8ee6cc6d642,keyword article,329,104,20919,324,326,1.55,2.2,0


## 5. Why ML beats a fixed rule here

A simple fixed rule might say, "Refresh every page that has not been updated for more than 180 days." However, age alone does not explain whether a page is likely to decline.

Refresh priority can depend on several connected signals, including content age, days since the last update, search impressions, clicks, sessions, click-through rate, search position and content type. Different combinations of these signals may produce different outcomes.

ML can learn relationships between several signals and produce one priority score for each page. A fixed rule normally uses only one or two manually selected thresholds.

The ranked output supports a real content action: a content editor reviews the highest-ranked pages and decides whether each page needs updating, rewriting, consolidation or no action.

The result will only be described as directional decision support. It will not prove that refreshing a page caused its performance to improve.

In [10]:
# Examine whether decline rates differ across combinations of page age and content type
group_check = (
    lane_df
    .groupby(
        ["age_tier", "content_type"],
        dropna=False
    )
    .agg(
        pages=("content_id", "count"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

# Keep groups large enough to make the comparison more meaningful
group_check = group_check[group_check["pages"] >= 30].copy()
group_check["decline_rate_pct"] = (
    group_check["decline_rate"] * 100
).round(2)

print("Decline rates vary across combinations of signals.")
print("This shows why one age rule may not work equally well for every page.")

group_check[
    [
        "age_tier",
        "content_type",
        "pages",
        "declining_pages",
        "decline_rate_pct"
    ]
].sort_values(
    "decline_rate_pct",
    ascending=False
).head(12)


Decline rates vary across combinations of signals.
This shows why one age rule may not work equally well for every page.


,age_tier,content_type,pages,declining_pages,decline_rate_pct
7,91-180,feedly article,79,57,72.15
8,91-180,keyword article,8385,5799,69.16
6,91-180,comparison article,253,174,68.77
4,31-90,keyword article,302,206,68.21
1,181-365,feedly article,271,180,66.42
2,181-365,keyword article,7266,4392,60.45
0,181-365,comparison article,113,64,56.64
5,365+,keyword article,5335,2279,42.72


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.